<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>




# Demo — Load and Explore Data

The ability to efficiently handle and explore data is paramount for machine learning projects. In this demo, we'll delve into techniques such as reading from and writing to Delta tables, recovering from mistakes with Delta time travel, computing statistics for machine learning insights, and visually exploring data using **Genie Code** for a comprehensive understanding of your datasets.

**Learning Objectives:**

By the end of this demo, you will be able to:

- **Load data into a Spark DataFrame**
   - Read a CSV file from a Unity Catalog volume with an explicit schema

- **Explore the dataset's structure and statistics**
   - Inspect the schema and preview the data with `display()`
   - Compute summary statistics with `DataFrame.summary()`
   - Compare built-in visualization tools with external Python libraries, and aggregate in Spark before plotting with matplotlib

- **Use Genie Code for ML-focused exploratory analysis**
   - Generate visualizations of the target distribution, missing values, and feature relationships
   - Interpret patterns that inform data preparation, such as class imbalance and missing values

- **Write data to a Delta table**
   - Save a DataFrame as a managed Delta table in Unity Catalog

- **Use Delta Lake time travel to recover from a mistake**
   - Simulate an accidental overwrite that drops columns
   - Review the table's change log with `DESCRIBE HISTORY`
   - Read an earlier version with `versionAsOf` to inspect or reproduce the original data
   - Restore the table to an earlier version with `RESTORE TABLE`
   - Explain how versioning supports reproducible ML datasets

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version ML V6</strong> - <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;" target="_blank">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless ML V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>


## Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this demo

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [ ]:
%run ../Includes/Classroom-Setup-1.2


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [ ]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Datasets Location: {DA.paths.datasets}")


## Read Dataset

In this step, we will read the `.csv` file from a shared volume and load it into a Spark DataFrame called `telco_df`.

In [ ]:
# Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn-missing' # CSV file name
dataset_path = f"{DA.paths.datasets.telco}/{shared_volume_name}/{csv_name}.csv" # Full path

schema_string = """
    customerID string,
    gender string,
    SeniorCitizen double,
    Partner string,
    Dependents string,
    tenure double,
    phoneService string,
    MultipleLines string,
    internetService string,
    OnlineSecurity string,
    OnlineBackup string,
    DeviceProtection string,
    TechSupport string,
    StreamingTV string,
    StreamingMovies string,
    Contract string,
    PaperlessBilling string,
    PaymentMethod string,
    MonthlyCharges double,
    TotalCharges double,
    Churn string
"""

telco_df = spark.read.csv(dataset_path, \
                          header=True,
                          schema=schema_string,
                          multiLine=True,
                          escape='"')

display(telco_df)


## Explore Data with Summary Stats

While using notebooks, you have various options to view summary statistics for a dataset. Some of these options are:

* Using Spark DataFrame's built-in method (e.g. `summary()`)
* Using Databricks' built-in data profiler and visualization editor
* Using Genie Code to generate targeted, ML-ready analyses

In this section we will go over Spark's built-in features for summarizing data. In the next section, we will use Genie Code for ML-focused visualization.



The first and simplest way is using Spark's `summary` function.

In [ ]:
# Display summary statistics with spark
display(telco_df.summary())



The **`display`** function not only facilitates the viewing of detailed dataframes but also serves as a powerful tool for visualizing our dataset according to individual preferences. Whether opting for a Pie Chart or a Bar Chart, this functionality allows for a more insightful exploration, uncovering dependencies and patterns within the dataframe features.

In [ ]:
display(telco_df)

### Integrated vs. External Visualization Tools

Databricks gives you two families of visualization tools. Most EDA uses both.

<table style="border-collapse: collapse; width: 100%; max-width: 860px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; margin: 16px 0;">
  <thead>
    <tr style="background-color: #1B3139;">
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Tool type</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Examples</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Best for</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Trade-off</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;"><strong style="color: #2272B4;">Integrated (built into the notebook)</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;"><code>display()</code> chart editor, <b>Data Profile</b> tab, AI/BI dashboards</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;">Fast, no-code exploration and sharing results with others</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;">Fewer statistical chart types and less control over styling</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;"><strong style="color: #2272B4;">External (Python libraries)</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;">matplotlib, seaborn, plotly</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;">Custom or statistical plots (box plots, pair plots, annotated charts) that you can version in code</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #E8E8E8;">You write (or generate) and maintain the plotting code</td>
    </tr>
  </tbody>
</table>

**Genie Code** bridges the two: it writes the external-library code for you from a natural-language prompt. Because generated code can vary between runs, the next cell shows a fixed example you can always fall back on.

In [ ]:
import matplotlib.pyplot as plt

# Aggregate in Spark, then plot the small result with matplotlib
churn_counts = telco_df.groupBy("Churn").count().toPandas()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(churn_counts["Churn"].astype(str), churn_counts["count"], color=["#2272B4", "#FF5F46"])
ax.set_title("Class balance: Churn")
ax.set_xlabel("Churn")
ax.set_ylabel("Customers")
plt.show()


## Data Visualization with Genie Code

In this section, you will use **Genie Code** to perform exploratory analysis of the Telco dataset from a machine learning perspective.

Instead of writing visualization code manually, you will use natural language prompts to generate charts and analyses that help you:

- Understand feature distributions  
- Identify missing value patterns  
- Analyze class imbalance  
- Explore relationships between features and the target variable  

Rather than creating one visualization at a time, you will provide a structured prompt that enables Genie Code to explore the dataset more holistically and generate multiple relevant insights.

This approach reflects how data scientists interact with AI-assisted workflows in real-world scenarios.

**Genie Code generates executable code directly in your notebook. You can review, run, and modify the generated code as needed.**


### How to Access Genie Code

To use Genie Code in Databricks:

1. Open the **Genie** panel on the right side of the notebook.  
![Databricks Genie Code](https://files.training.databricks.com/binder/prod_main/data-preparation-for-machine-learning-en_us-4.0.0/images/20261111T171021Z/Data%20Preparation%20for%20Machine%20Learning/Includes/images/1.1_Genie_intro.png)

2. You will see the **Genie Code** panel like this- 
   <p align="center">
     <img src="https://files.training.databricks.com/binder/prod_main/data-preparation-for-machine-learning-en_us-4.0.0/images/20261111T171021Z/Data Preparation for Machine Learning/Includes/images/genie_code_chat.png"
          alt="The Agent option in the Genie Code mode drop-down"
          title="The Agent option in the Genie Code mode drop-down"
          width="600"/>
   </p>

3. Use prompts such as:
   - "Plot the distribution of the `Churn` column in `telco_df`"
   - "Show a heatmap of missing values in `telco_df`"
   - "Compare `MonthlyCharges` across `Churn` categories using a boxplot"

Genie Code will generate executable code directly in the notebook, which you can run and refine as needed.


### Best Practices for Genie Code Prompts

To get reliable and useful results from Genie Code, write prompts that clearly specify:

- **DataFrame name** — specify which dataset to use (for example, `telco_df`)
- **Column names** — explicitly mention the columns to analyze
- **Chart type** — define the visualization if you have a preference
- **Target column** — identify the label for ML-focused analysis
- **Objective** — clearly state the question you want to answer

**Example:**
> "Using `telco_pdf`, create a countplot of the `Churn` column to show class balance.  
> Use seaborn or matplotlib. Add a title and axis labels."


### Important Considerations

<div style="border-left: 4px solid #f57c00; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">⚠️</span>
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Always Review Generated Code</strong>
            <p style="margin: 8px 0 0 0; color: #333;">
                Genie Code can accelerate development, but you remain responsible for reviewing the generated code and verifying correctness before running it.
            </p>
        </div>
    </div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">🛠️</span>
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Troubleshooting the Genie Code Panel</strong>
            <p style="margin: 8px 0 0 0; color: #333;">
                If you do not see the Genie Code panel, try the following:
            </p>
            <ul style="margin: 8px 0 0 18px; color: #333;">
                <li>Confirm you are using a supported compute environment (Serverless ML6 for this course)</li>
                <li>Ensure Genie Code is enabled in your workspace</li>
                <li>Refresh your browser or reopen the notebook</li>
            </ul>
            <p style="margin: 10px 0 0 0; color: #333;">
                <strong>Note:</strong> Your instructor may have already configured the required workspace settings for this course.
            </p>
        </div>
    </div>
</div>


### What to Analyze

During this analysis, you will use Genie Code to explore the dataset and automatically generate both visualizations and insights.

Focus on understanding:

- The distribution of the target variable (`Churn`) and whether it is balanced  
- Which columns contain missing values and may require imputation  
- Relationships between features and the target variable  
- Patterns in the data that may influence model performance  


**Use Genie Code to Explore the Dataset**

**_Use the prompt below with Genie Code to perform a complete exploratory analysis and generate insights:_**

<button id="copy-genie-btn-task2" onclick="copyGenieTask2()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600;">
Copy Genie Code Prompt
</button>

<pre id="copy-genie-task2" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre-wrap;">
You are a data analyst working in a Databricks notebook.

We have a Spark DataFrame called `telco_df`.

Perform an exploratory data analysis focused on machine learning readiness.

Steps:
1. Convert `telco_df` to a pandas DataFrame called `telco_pdf`

2. Analyze and visualize:
   - Distribution of the target variable `Churn` (class balance with counts and percentages)
   - Missing value percentage across columns
   - Relationship between key numeric features (`tenure`, `MonthlyCharges`, `TotalCharges`) and `Churn`
   - Distribution of important categorical features (`Contract`, `PaymentMethod`, `InternetService`) grouped by `Churn`

3. Create appropriate visualizations using matplotlib or seaborn
4. Add titles and labels to all charts
5. Keep the code clean and readable
6. Generate multiple code cells if needed

After generating the charts, provide a clear summary that answers:

- Is the dataset balanced or imbalanced?
- Which features appear to influence churn the most?
- What data quality issues should be addressed before modeling?
</pre>

<script>
function copyGenieTask2() {
  const textEl = document.getElementById("copy-genie-task2");
  const buttonEl = document.getElementById("copy-genie-btn-task2");

  if (!textEl || !buttonEl) return;

  const text = textEl.innerText;
  const originalText = "Copy Genie Code Prompt";

  function showCopiedState() {
    buttonEl.innerText = "✅ Copied";
    setTimeout(() => {
      buttonEl.innerText = originalText;
    }, 2000);
  }

  function fallbackCopy(text) {
    const ta = document.createElement("textarea");
    ta.value = text;
    ta.style.position = "fixed";
    ta.style.left = "-9999px";
    document.body.appendChild(ta);
    ta.focus();
    ta.select();

    try {
      document.execCommand("copy");
      showCopiedState();
    } catch (err) {
      alert("Could not copy automatically. Please copy manually.");
    } finally {
      document.body.removeChild(ta);
    }
  }

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => showCopiedState())
      .catch(() => fallbackCopy(text));
  } else {
    fallbackCopy(text);
  }
}
</script>


## Write Data to Delta Table

In this step, we will persist the dataset as a **Delta table**.

Storing data in Delta format enables features such as:
- ACID transactions  
- schema enforcement  
- time travel for versioning  

We will save the dataset as a managed table in the default catalog and schema.

In [ ]:
# Specify table name
table_name_bronze = "telco_missing_bronze"

# Write DataFrame to Delta table
telco_df.write.saveAsTable(table_name_bronze)

print(f"Table created: {table_name_bronze}")


## Time Travel with Delta Lake

In this section, we will explore **Delta Lake time travel**, which allows you to access previous versions of a table.

This capability enables you to:
- Reproduce past results  
- Audit data changes  
- Recover from unintended updates  

Time travel is especially valuable in machine learning workflows, where consistent and versioned data is critical for experimentation.

**Note:** While versioning improves reproducibility, it may introduce challenges in long-running ML workflows if data retention policies limit access to older versions.


### Simulate an Accidental Change

To demonstrate time travel, we will simulate a common scenario where columns are unintentionally removed from a table.

In [ ]:
# Drop columns and overwrite table
to_drop_wrong = ["gender", "SeniorCitizen"]
telco_dropped_df = telco_df.drop(*to_drop_wrong)

telco_dropped_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(table_name_bronze)


### Review the Table History

What do we do if we drop a column by mistake?

Every write to a Delta table creates a new version. Let's use the **`DESCRIBE HISTORY`** SQL command to review the history of changes and find the version where `gender` and `SeniorCitizen` were still part of our dataset.

In [ ]:
%sql
DESCRIBE HISTORY telco_missing_bronze

Examine the schema to validate whether columns were dropped, specifically verifying if the `gender` and `SeniorCitizen` columns have been removed.

In [ ]:
spark.table(table_name_bronze).printSchema()

You can also use the SQL `DESCRIBE` command to view the table schema.

In [ ]:
%sql
DESCRIBE `telco_missing_bronze`;

### Read a Previous Version
As we can see in the **`operation`** column, version 1 overwrote the table and removed the columns. Version 0 still has all the original columns.

Reading a previous version with `versionAsOf` (or `timestampAsOf`) returns the data as it was at that version. It **does not change the table**. This is how you reproduce the exact data a model was trained on.

Read version 0 and inspect its schema to confirm that `gender` and `SeniorCitizen` are present in that version.

In [ ]:
telco_df_v0 = (
  spark.read
      .option("versionAsOf", 0)
      .table(table_name_bronze)
)

telco_df_v0.printSchema()

### Restore the Table to a Previous Version

Reading version 0 gave us the original data, but the table itself still has the columns missing. Anyone who queries `telco_missing_bronze` gets the broken version.

To fix the table, use **`RESTORE TABLE`**. It rolls the table back to version 0 by writing a **new** version, so the history is kept and the restore can itself be undone.

In [ ]:
%sql
RESTORE TABLE telco_missing_bronze TO VERSION AS OF 0

Review the history again. The latest version shows the **`RESTORE`** operation, and versions 0 and 1 are still there.

In [ ]:
%sql
DESCRIBE HISTORY telco_missing_bronze

Confirm that the restored table has all of its original columns again.

In [ ]:
spark.table(table_name_bronze).printSchema()

## Conclusion

In this demo, you explored key techniques for data management and exploratory analysis using Delta Lake and Spark.

You were able to:
- Load and inspect a dataset using Spark DataFrames  
- Compute summary statistics to understand data characteristics  
- Use **Genie Code** to generate ML-focused visualizations  
- Write data to a Delta table in Unity Catalog
- Use time travel to review a table's history, read a previous version, and restore the table after an accidental change

These capabilities provide a strong foundation for preparing data and building reliable machine learning workflows.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>

